# Llama 3.2 3B — Edge Benchmark Analysis
**Model:** Llama 3.2 3B Instruct Q4_K_M  
**Backend:** CPU-only (llama.cpp, 12 threads)  
**Rationale for selection:** Fastest throughput and lowest memory footprint vs Phi-3 Mini in Phase 1 head-to-head.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#0d0f14', 'axes.facecolor': '#13161e',
    'axes.edgecolor': '#252a38', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#64748b', 'ytick.color': '#64748b',
    'text.color': '#e2e8f0', 'grid.color': '#252a38',
    'grid.linewidth': 0.8, 'font.family': 'monospace',
    'axes.titlesize': 12, 'axes.labelsize': 10,
})

ACCENT  = '#5af0c4'
PLOTS   = Path('../results/plots')
PLOTS.mkdir(parents=True, exist_ok=True)

perf = pd.read_csv('../results/raw/performance_phase1.csv')
ppl  = pd.read_csv('../results/raw/perplexity_phase1.csv')

display(perf)
display(ppl)

## 1. Throughput

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Llama 3.2 3B Q4_K_M — Throughput (CPU, 12 threads)', fontweight='bold')

metrics = [('pp512_mean', 'pp512_std', 'Prompt Processing (pp512)', 't/s', 130),
           ('tg128_mean', 'tg128_std', 'Token Generation (tg128)', 't/s', 25)]

for ax, (mean_col, std_col, title, ylabel, ylim) in zip(axes, metrics):
    val, err = perf[mean_col].values[0], perf[std_col].values[0]
    bar = ax.bar(['Llama 3.2 3B'], [val], yerr=[err], capsize=8,
                 color=ACCENT, edgecolor='none', width=0.35,
                 error_kw=dict(ecolor='#64748b', lw=1.5))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, ylim)
    ax.grid(axis='y', alpha=0.4)
    ax.text(0, val + err + ylim*0.02, f'{val:.2f}', ha='center',
            fontsize=12, fontweight='bold', color='#e2e8f0')

plt.tight_layout()
plt.savefig(PLOTS / 'llama_throughput.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

## 2. Memory Breakdown

In [ ]:
# From llama.cpp memory breakdown logs
labels  = ['Model\nweights', 'KV cache', 'Compute\nbuffer']
values  = [1918, 224, 262]  # MiB
colors  = ['#3b6978', ACCENT, '#475569']

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Bar chart
axes[0].bar(labels, values, color=colors, edgecolor='none', width=0.5)
axes[0].set_ylabel('MiB')
axes[0].set_title('Memory by Component', fontweight='bold')
axes[0].grid(axis='y', alpha=0.4)
for i, v in enumerate(values):
    axes[0].text(i, v+10, f'{v} MiB', ha='center', fontsize=9, fontweight='bold')

# Pie
wedge_props = dict(edgecolor='#0d0f14', linewidth=2)
axes[1].pie(values, labels=labels, colors=colors, autopct='%1.0f%%',
            wedgeprops=wedge_props, textprops={'color': '#e2e8f0', 'fontsize': 9})
axes[1].set_title(f'Peak RAM: {perf["peak_rss_mib"].values[0]:.0f} MiB total', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS / 'llama_memory.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

## 3. Perplexity — Per-Chunk Profile

In [ ]:
chunks = [5.8596,7.4719,8.8219,9.1851,9.5743,9.9819,10.6463,11.3612,12.4467,12.6558,
          12.7377,13.0210,13.7183,13.0090,12.9041,12.3457,12.1186,12.3250]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(range(1, len(chunks)+1), chunks, 'o-', color=ACCENT, linewidth=2, markersize=5)
ax.fill_between(range(1, len(chunks)+1), chunks, alpha=0.1, color=ACCENT)
ax.axhline(y=ppl['ppl'].values[0], color=ACCENT, linestyle='--', alpha=0.6, linewidth=1.2,
           label=f'Final PPL = {ppl["ppl"].values[0]}')
ax.set_xlabel('Chunk index')
ax.set_ylabel('Perplexity')
ax.set_title('Per-Chunk Perplexity — Llama 3.2 3B (Wikitext-2, ctx=512)', fontweight='bold')
ax.legend(fontsize=9, framealpha=0.3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS / 'llama_perplexity.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

print(f"Note: PPL rises steeply through chunks 1-13 before stabilising.")
print(f"This indicates the model is slower to build up context on diverse Wikipedia text.")

## 4. Interpretation

### Why Llama 3.2 3B was chosen
Llama 3.2 3B uses **Grouped Query Attention (GQA)** with 8 KV heads against 24 query heads. This slashes KV cache memory to 224 MiB at ctx-size 512 (vs 768 MiB for Phi-3 Mini), which is the primary driver of both its speed and memory advantage.

| Metric | Value | vs Phi-3 Mini |
|--------|-------|----------------|
| pp512 | 110.43 t/s | +52% faster |
| tg128 | 19.22 t/s | +20% faster |
| Peak RAM | 2,114 MiB | −18% |
| KV cache | 224 MiB | −71% |

### Perplexity caveat
Llama 3.2 3B has a higher perplexity (12.33) than Phi-3 Mini (6.82) on Wikitext-2 at ctx-size 512. The per-chunk curve shows a gradual rise before stabilising — suggesting it takes longer to accumulate useful context. This trade-off is acceptable for applications where throughput and memory headroom are the priority.

### Next steps
1. Extend context size tests (2048, 4096) — Llama 3.2's KV advantage grows with context length
2. Run GSM8K and HumanEval to verify task-level quality
3. Test Q5_K_M quantisation for a quality/speed mid-point
4. Profile at different thread counts to find the optimal CPU saturation point